<div align="center">
<img src="https://poorit.in/image.png" alt="Poorit" width="40" style="vertical-align: middle;"> <b>LPU — BACKEND & GENERATIVE AI</b>

## Final Exercise — Build the Control Panel

**Lovely Professional University**
*Backend & Generative AI · Poorit Technologies*

</div>

---

### What You'll Build

1. Wrap a model in a **chat UI** with one function and `gr.ChatInterface`
2. Add a **system-prompt box** and change your bot's personality mid-conversation
3. Add a **temperature slider** and watch the same question answer differently
4. Add a **model dropdown** — and swap provider without touching your code
5. Get a **public link** you can open on your phone

> **The last thing you build this week, and the smallest.** No RAG, no agents, no vector
> store — one function, three widgets, and every knob from the week on one screen.
>
> **You need an OpenAI API key.** A Gemini key is optional and only used by the last step.

---

## 1. Setup

Run these three cells. Nothing to write yet.

In [ ]:
# PROVIDED - just run this cell.
!pip install -q litellm "gradio>=6"

In [ ]:
# PROVIDED - just run this cell.
import os
from getpass import getpass

import gradio as gr
from litellm import completion

print("gradio", gr.__version__)      # this notebook is written for gradio 6+

In [ ]:
# PROVIDED - just run this cell.
os.environ["OPENAI_API_KEY"] = getpass("OpenAI API Key: ")
print("key loaded")

---

## 2. Build it

**The tools you have:**

| What | How |
|---|---|
| call a model | `completion(model=..., messages=[...], temperature=...)` |
| read the reply | `response.choices[0].message.content` |
| the model string | `"openai/gpt-4o-mini"` — LiteLLM wants the **provider prefix** |
| a message | `{"role": "system" \| "user" \| "assistant", "content": "..."}` |
| a chat UI | `gr.ChatInterface(fn).launch()` |
| extra widgets | `gr.ChatInterface(fn, additional_inputs=[...])` |

**How `gr.ChatInterface` calls your function:**

```
  fn(message, history)                        <- with no additional_inputs
  fn(message, history, extra1, extra2, ...)   <- one extra argument per widget, in order
```

`history` arrives as a list of `{"role": ..., "content": ...}` dicts — **the exact shape the API
wants**, so it drops straight into your messages list. Gradio keeps the conversation for you.

> ⚠️ **Do not pass `type="messages"`.** You will see it in older tutorials and in Day-2 material.
> Gradio 6 removed that argument — messages is now the only format — and passing it raises
> `TypeError: unexpected keyword argument 'type'`.

Everything from here is yours to write.

In [ ]:
# STEP 1 - Write chat(message, history) and return a reply.
#
#   1. build a list: a system message, then *history, then the new user message
#   2. call completion(model="openai/gpt-4o-mini", messages=<that list>)
#   3. return the reply TEXT (not the whole response object)
#
# Test it as a plain function first - no UI yet:
#     print(chat("hello", []))

In [ ]:
# STEP 2 - Launch it.
#
#   gr.ChatInterface(chat).launch()
#
# Send it three messages. Then ask "what did I just say?" - it remembers, because
# Gradio hands you the whole history every turn. The model itself remembers nothing.

---

## 3. Add the panel

One widget at a time. Re-launch after each — seeing each control appear is the point.

⚠️ **Argument order matters.** The widgets arrive in the same order you list them, straight after
`history`.

In [ ]:
# STEP 3 - A system-prompt box.
#
#   1. change the signature to  chat(message, history, system)
#   2. use `system` as the system message instead of your hard-coded one
#   3. launch with:
#        gr.ChatInterface(chat, additional_inputs=[gr.Textbox("You are a helpful assistant.",
#                                                             label="System prompt")]).launch()
#
# Then: start a normal conversation, CHANGE the system prompt to something absurd
# ("answer only in questions"), and keep chatting. What happened to the earlier replies?

In [ ]:
# STEP 4 - A temperature slider.
#
#   1. signature becomes  chat(message, history, system, temperature)
#   2. pass it through:  completion(..., temperature=temperature)
#   3. add to additional_inputs:
#        gr.Slider(0, 2, value=0.7, step=0.1, label="Temperature")
#
# Then: set it to 0 and ask the SAME question three times. Now set it to 1.8 and
# ask the same question three times again. Write down what changed.

In [ ]:
# STEP 5 - A model dropdown. This is the one worth doing.
#
#   1. signature becomes  chat(message, history, system, temperature, model)
#   2. pass it through:   completion(model=model, ...)
#   3. add to additional_inputs:
#        gr.Dropdown(["openai/gpt-4o-mini", "openai/gpt-4o"],
#                    value="openai/gpt-4o-mini", label="Model")
#
# Switch models mid-conversation and ask the same question. Your function did not
# change. Not one line. That is the whole argument for LiteLLM.

---

## 4. Stretch — only if steps 1–5 work

**A · A different company entirely.** Get a free key at
[aistudio.google.com](https://aistudio.google.com/apikey), then:

```python
os.environ["GEMINI_API_KEY"] = getpass("Gemini API Key: ")
```

Add `"gemini/gemini-2.0-flash"` to your dropdown and switch to it mid-conversation.
**Different company, different model, same function.** Nothing in your code knows.

**B · Share it.** `launch(share=True)` gives you a public link for 72 hours. Open it on your
phone and send it to someone.

**C · Stream it.** Turn `chat` into a generator — add `stream=True` to `completion(...)`, then
`yield` the reply as it grows instead of `return`-ing it at the end. Gradio renders a generator
token by token with no other change.

**D · Give it examples.** `gr.ChatInterface(chat, examples=["Explain recursion", "..."], ...)`
puts clickable starter prompts under the box.

---

**When it misbehaves:**

| What you see | What it means |
|---|---|
| `TypeError: ... unexpected keyword argument 'type'` | You passed `type="messages"`. Delete it — Gradio 6 does not take it. |
| `TypeError: chat() takes 2 positional arguments but 5 were given` | You added widgets to `additional_inputs` but did not add the matching parameters to your function. |
| The widgets arrive in the wrong variables | `additional_inputs` order **is** your parameter order. Line them up. |
| `AuthenticationError` | Re-run the key cell. For Gemini, the variable is `GEMINI_API_KEY`. |
| `BadRequestError: LLM Provider NOT provided` | Missing the provider prefix — `"gpt-4o-mini"` should be `"openai/gpt-4o-mini"`. |
| The bot ignores your system prompt | You built the messages list in the wrong order, or dropped `*history`. System first, then history, then the new message. |
| Replies are identical at temperature 1.8 | You forgot to pass `temperature=` through to `completion`. |
| The old UI is still showing | Re-run the launch cell. In Colab, the previous one keeps its own port. |

---

### ✅ What you practised

| Idea | The one-liner |
|---|---|
| **One function is a UI** | `gr.ChatInterface(fn)` — message box, bubbles, scrolling, free |
| **Memory is a list** | Gradio hands you `history` every turn; the model remembers nothing |
| **`additional_inputs`** | one extra function argument per widget, in order |
| **System prompt** | changes behaviour *retroactively* — the whole history is resent every turn |
| **Temperature** | 0 for the same answer every time, high for variety |
| **The provider prefix** | `openai/…`, `gemini/…` — one string is the entire switch |
| **LiteLLM's argument** | your function never learns which company answered |

**The one thing to take away:** you changed model, provider, personality and randomness from a
web page — and the function underneath never changed. That is what a good interface buys you, and
it is the same reason FastAPI can validate a request you never wrote a check for.